**Proposition 1: Customers Who Bought Both Bikes and Accessories**  
  

**Functional Spec:**

Identify customers who have purchased at least one \*Bike\* product \*\*and\*\* at least one \*Accessory\* product. <span style="color: var(--vscode-foreground);">This helps analyze cross-category purchasing behavior — ideal for targeted marketing and bundling strategies.</span>

**SQL Concept Used:** INTERSECT  
  

**Expected Output:**

CustomerID and count of distinct orders per segment (i.e., customers appearing in both sets).

In [20]:
-- Customers who bought a Bike
SELECT DISTINCT soh.CustomerID
FROM Sales.SalesOrderHeader AS soh
JOIN Sales.SalesOrderDetail AS sod ON soh.SalesOrderID = sod.SalesOrderID
JOIN Production.Product AS p ON sod.ProductID = p.ProductID
JOIN Production.ProductSubcategory AS ps ON p.ProductSubcategoryID = ps.ProductSubcategoryID
JOIN Production.ProductCategory AS pc ON ps.ProductCategoryID = pc.ProductCategoryID
WHERE pc.Name = 'Bikes'

INTERSECT

-- Customers who bought an Accessory
SELECT DISTINCT soh.CustomerID
FROM Sales.SalesOrderHeader AS soh
JOIN Sales.SalesOrderDetail AS sod ON soh.SalesOrderID = sod.SalesOrderID
JOIN Production.Product AS p ON sod.ProductID = p.ProductID
JOIN Production.ProductSubcategory AS ps ON p.ProductSubcategoryID = ps.ProductSubcategoryID
JOIN Production.ProductCategory AS pc ON ps.ProductCategoryID = pc.ProductCategoryID
WHERE pc.Name = 'Accessories';

(6863 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:01.735

CustomerID
15652
14324
19897
28387
25708
17003
15675
17218
25493
29761


**Proposition 2: Combine Lists of All Current and Former Employees**

**Statement:**

Generate a unified record of all employees — both current and former — to support HR audits and workforce history reporting.

**Functional Specification:**

Produce a single list of employees with status determined via EmployeeDepartmentHistory.EndDate (NULL =\> current; NOT NULL =\> former).

**SQL Concept Used:** UNION

**Expected Output:**

Distinct rows: BusinessEntityID, JobTitle, Status ('Current'/'Former').

In [16]:
-- Current employees (EmployeeDepartmentHistory.EndDate IS NULL)
SELECT DISTINCT e.BusinessEntityID,
       e.JobTitle,
       'Current Employee' AS EmploymentStatus
FROM HumanResources.Employee AS e
JOIN HumanResources.EmployeeDepartmentHistory AS edh
  ON e.BusinessEntityID = edh.BusinessEntityID
WHERE edh.EndDate IS NULL

UNION

-- Former employees (EmployeeDepartmentHistory.EndDate IS NOT NULL)
SELECT DISTINCT e.BusinessEntityID,
       e.JobTitle,
       'Former Employee' AS EmploymentStatus
FROM HumanResources.Employee AS e
JOIN HumanResources.EmployeeDepartmentHistory AS edh
  ON e.BusinessEntityID = edh.BusinessEntityID
WHERE edh.EndDate IS NOT NULL;

(295 rows affected)

Total execution time: 00:00:01.601

BusinessEntityID,JobTitle,EmploymentStatus
1,Chief Executive Officer,Current Employee
2,Vice President of Engineering,Current Employee
3,Engineering Manager,Current Employee
4,Senior Tool Designer,Current Employee
4,Senior Tool Designer,Former Employee
5,Design Engineer,Current Employee
6,Design Engineer,Current Employee
7,Research and Development Manager,Current Employee
8,Research and Development Engineer,Current Employee
9,Research and Development Engineer,Current Employee


**Proposition 3: Find Products That Were Sold but Are No Longer Manufactured**

**Statement:**

Identify all products that were once sold to customers but are no longer available for manufacturing or sales.

**Functional Specification:**

Using the \`INTERSECT\` operator, this query identifies products that appear in the sales order history and also have a \`SellEndDate\` recorded — meaning they were discontinued after being sold.

**SQL Concept Used:** INTERSECT

**Why It’s Special:**

It highlights legacy or end-of-life products that can still affect customer support, spare parts inventory, and after-sales service strategies.

**Expected Output:**

Distinct list of \`ProductID\` values for discontinued yet previously sold products.

In [3]:
SELECT DISTINCT ProductID
FROM Sales.SalesOrderDetail
INTERSECT
SELECT ProductID
FROM Production.Product
WHERE SellEndDate IS NOT NULL;

(83 rows affected)

Total execution time: 00:00:00.040

ProductID
709
710
725
726
727
729
730
732
733
741


**Proposition 4: Identify Employees Who Have Never Made a Sale**

**Statement:**

Determine which sales employees exist in the HR database but have not completed any recorded sales transactions.  
  

**Functional Specification:**

This query uses the \`EXCEPT\` operator to subtract the set of employees who appear in sales order headers from the total list of employees holding sales-related job titles.  
  

**SQL Concept Used:** EXCEPT  
  

**Why It’s Special:**

It supports performance review analysis and helps sales management identify underutilized or misclassified employees.  
  

**Expected Output:**

List of employee names and IDs who are tagged as sales staff but have zero sales in their record.

In [4]:
SELECT e.BusinessEntityID, p.FirstName, p.LastName
FROM HumanResources.Employee AS e
JOIN Person.Person AS p ON e.BusinessEntityID = p.BusinessEntityID
WHERE e.JobTitle LIKE '%Sales%'
EXCEPT
SELECT DISTINCT soh.SalesPersonID, p2.FirstName, p2.LastName
FROM Sales.SalesOrderHeader AS soh
JOIN Person.Person AS p2 ON soh.SalesPersonID = p2.BusinessEntityID
WHERE soh.SalesPersonID IS NOT NULL;


(1 row affected)

Total execution time: 00:00:00.205

BusinessEntityID,FirstName,LastName
273,Brian,Welcker


**Proposition 5: Products Sold in 2013 But Not in 2014**

**Functional Spec:**

Find products that were sold in 2013 but not sold in 2014 — great for analyzing discontinued or declining items.

**SQL Concept Used:** EXCEPT

**Expected Output:**

List of ProductID and ProductName for products that disappeared from the sales record between 2013 and 2014.

In [21]:
-- Products sold in 2013
SELECT DISTINCT p.ProductID, p.Name
FROM Sales.SalesOrderHeader AS soh
JOIN Sales.SalesOrderDetail AS sod ON soh.SalesOrderID = sod.SalesOrderID
JOIN Production.Product AS p ON sod.ProductID = p.ProductID
WHERE YEAR(soh.OrderDate) = 2013

EXCEPT

-- Products sold in 2014
SELECT DISTINCT p.ProductID, p.Name
FROM Sales.SalesOrderHeader AS soh
JOIN Sales.SalesOrderDetail AS sod ON soh.SalesOrderID = sod.SalesOrderID
JOIN Production.Product AS p ON sod.ProductID = p.ProductID
WHERE YEAR(soh.OrderDate) = 2014;

(58 rows affected)

Total execution time: 00:00:02.029

ProductID,Name
902,"LL Touring Frame - Yellow, 58"
919,"LL Mountain Frame - Silver, 48"
816,ML Mountain Front Wheel
942,"ML Mountain Frame-W - Silver, 38"
819,ML Road Front Wheel
770,"Road-650 Black, 52"
839,"HL Road Frame - Black, 48"
833,"ML Road Frame-W - Yellow, 40"
802,LL Fork
825,HL Mountain Rear Wheel


**Proposition 6: Compare Products Sold Online vs. In-Store**

**Statement:**

Differentiate between products sold exclusively online and those sold exclusively in-store.

**Functional Specification:**

By applying the \`EXCEPT\` operator twice (and optionally combining with \`UNION\`), we isolate products appearing only in online (\`OnlineOrderFlag = 1\`) or in-store (\`OnlineOrderFlag = 0\`) orders.

**SQL Concept Used:** EXCEPT, UNION

**Why It’s Special:**

It’s a market insights tool that helps the business understand product performance across sales channels and identify where exclusivity drives demand.

**Expected Output:**

Two lists:

1\. Products sold only online.  

2\. Products sold only in-store.

In [12]:
-- Products sold only online
SELECT DISTINCT ProductID
FROM Sales.SalesOrderHeader soh
JOIN Sales.SalesOrderDetail sod ON soh.SalesOrderID = sod.SalesOrderID
WHERE soh.OnlineOrderFlag = 1
EXCEPT
SELECT DISTINCT ProductID
FROM Sales.SalesOrderHeader soh
JOIN Sales.SalesOrderDetail sod ON soh.SalesOrderID = sod.SalesOrderID
WHERE soh.OnlineOrderFlag = 0;

-- Products sold only in-store
SELECT DISTINCT ProductID
FROM Sales.SalesOrderHeader soh
JOIN Sales.SalesOrderDetail sod ON soh.SalesOrderID = sod.SalesOrderID
WHERE soh.OnlineOrderFlag = 0
EXCEPT
SELECT DISTINCT ProductID
FROM Sales.SalesOrderHeader soh
JOIN Sales.SalesOrderDetail sod ON soh.SalesOrderID = sod.SalesOrderID
WHERE soh.OnlineOrderFlag = 1;

(16 rows affected)

(136 rows affected)

Total execution time: 00:00:01.030

ProductID
879
922
928
713
931
882
923
871
934
932


ProductID
925
902
710
733
856
756
802
825
948
919


**Proposition 7: Employees Who Worked in Both 'Sales' and 'Marketing' Departments**

**Functional Specification:**

Identify employees who have been assigned to both \*Sales\* and \*Marketing\* departments at different times. <span style="color: var(--vscode-foreground);">This reveals multi-department experience or internal mobility — valuable for HR analytics.</span>

**SQL Concept Used:** INTERSECT

**Expected Output:**

List of BusinessEntityIDs (employees) who have worked in both Sales and Marketing.

In [22]:
-- Employees who have worked in the Sales department
SELECT DISTINCT edh.BusinessEntityID
FROM HumanResources.EmployeeDepartmentHistory AS edh
JOIN HumanResources.Department AS d ON edh.DepartmentID = d.DepartmentID
WHERE d.Name = 'Sales'

INTERSECT

-- Employees who have worked in the Marketing department
SELECT DISTINCT edh.BusinessEntityID
FROM HumanResources.EmployeeDepartmentHistory AS edh
JOIN HumanResources.Department AS d ON edh.DepartmentID = d.DepartmentID
WHERE d.Name = 'Marketing';

(0 rows affected)

Total execution time: 00:00:00.512

BusinessEntityID


**Proposition 8: Combine Vendor and Customer Contact Lists**

**Statement:**

Develop a comprehensive master contact list combining all vendors and customers for communication and CRM integration.

**Functional Specification:**

The \`UNION\` operator merges contact data from vendors (\`Purchasing.Vendor\`) and customers (\`Sales.Customer\`), ensuring a non-duplicated and unified dataset.

**SQL Concept Used:** UNION

**Why It’s Special:**

This enhances communication pipelines and simplifies mass outreach for surveys, feedback, or announcements.

**Expected Output:**

Distinct combined list of contact names and types (\`Vendor\` or \`Customer\`).

In [9]:
SELECT v.BusinessEntityID, p.FirstName, p.LastName, 'Vendor' AS ContactType
FROM Purchasing.Vendor v
JOIN Person.Person p ON v.BusinessEntityID = p.BusinessEntityID
UNION
SELECT c.CustomerID AS BusinessEntityID, p.FirstName, p.LastName, 'Customer' AS ContactType
FROM Sales.Customer c
JOIN Person.Person p ON c.PersonID = p.BusinessEntityID;

(19119 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:02.332

BusinessEntityID,FirstName,LastName,ContactType
29485,Catherine,Abel,Customer
29486,Kim,Abercrombie,Customer
29487,Humberto,Acevedo,Customer
29484,Gustavo,Achong,Customer
29488,Pilar,Ackerman,Customer
28866,Aaron,Adams,Customer
13323,Adam,Adams,Customer
21139,Alex,Adams,Customer
29170,Alexandra,Adams,Customer
19419,Allison,Adams,Customer


**Proposition 9: Find Employees Who Are Also Customers**

**Statement:**

Identify employees who also appear in the customer database, revealing internal buyers or loyalty program participants.

**Functional Specification:**

This query finds overlapping individuals between the \`HumanResources.Employee\` and \`Sales.Customer\` datasets using the \`INTERSECT\` operator.

**SQL Concept Used:** INTERSECT

**Why It’s Special:**

It offers insight into employee purchasing behavior — useful for discount programs and internal incentives.

**Expected Output:**

Employee names that exist in both the employee and customer records.

In [8]:
SELECT p.FirstName, p.LastName
FROM Person.Person p
JOIN HumanResources.Employee e ON p.BusinessEntityID = e.BusinessEntityID
INTERSECT
SELECT p2.FirstName, p2.LastName
FROM Person.Person p2
JOIN Sales.Customer c ON p2.BusinessEntityID = c.PersonID;

(120 rows affected)

Total execution time: 00:00:00.942

FirstName,LastName
Alan,Brewer
Alejandro,McGuel
Amy,Alberts
Andrew,Cencini
Andrew,Hill
Andy,Ruth
Anibal,Sousa
Barbara,Decker
Baris,Cetinok
Barry,Johnson


**Proposition 10: Identify Products Never Sold**

**Statement:**

Determine which products exist in the catalog but have never been sold, to identify dead stock or catalog redundancies.

**Functional Specification:**

Return product catalog entries that never appear in any order detail. Use \`EXCEPT\` with properly qualified columns.

**SQL Concept Used:** EXCEPT

**Expected Output:**

ProductID and Name for products that were never included in any \`Sales.SalesOrderDetail\`.

In [19]:
-- All products in catalog
SELECT p.ProductID, p.Name
FROM Production.Product AS p

EXCEPT

-- Products that appear in order details (qualify columns to avoid ambiguity)
SELECT DISTINCT p2.ProductID, p2.Name
FROM Sales.SalesOrderDetail AS sod
JOIN Production.Product AS p2 ON sod.ProductID = p2.ProductID;

(238 rows affected)

Total execution time: 00:00:00.301

ProductID,Name
1,Adjustable Race
3,BB Ball Bearing
2,Bearing Ball
316,Blade
324,Chain Stays
322,Chainring
320,Chainring Bolts
321,Chainring Nut
505,Cone-Shaped Race
323,Crown Race
